Hipótesis 1: ¿Los hijos aseguran el viaje? Escribe una consulta usando CASE WHEN que clasifique a la reserva como "Con Hijos" (si children > 0 o babies > 0) o "Sin Hijos". Agrupa por esta nueva categoría y calcula la tasa de cancelación (puedes sacar el promedio - AVG - de la columna is_canceled). ¿Qué segmento cancela más?

In [0]:
%sql
USE CATALOG cursodataacademy;
USE SCHEMA hotel_db;

In [0]:
%sql
SELECT CASE WHEN (children > 0 AND NOT isnan(children)) OR babies > 0
            THEN 'Con Hijos' ELSE 'Sin Hijos' END AS segmento,
       COUNT(*) AS total_reservas,
       ROUND(AVG(is_canceled), 4) AS tasa_cancelacion
FROM tb_reservas
GROUP BY CASE WHEN (children > 0 AND NOT isnan(children)) OR babies > 0
              THEN 'Con Hijos' ELSE 'Sin Hijos' END;

Conclusión: Las reservas sin hijos cancelan más (37.22%) que las que tienen hijos (34.92%), una diferencia de solo 2.3 puntos porcentuales.
Los hijos reducen levemente el riesgo de cancelar, pero no aseguran el viaje: más de 1 de cada 3 reservas con hijos igual se cancela.
Además, solo el 7.8% de las reservas (9,332) incluye hijos, por lo que este criterio por sí solo no basta para definir políticas de depósito.

Hipótesis 2: El impacto de las solicitudes especiales. Agrupa por la columna total_of_special_requests y calcula la tasa de cancelación (AVG(is_canceled)) para cada cantidad de solicitudes (0, 1, 2, etc.). Ordena por la cantidad de solicitudes de forma ascendente. ¿A medida que el cliente pide más cosas, cancela más o cancela menos?


In [0]:
%sql
SELECT total_of_special_requests,
       COUNT(*) AS total_reservas,
       ROUND(AVG(is_canceled), 4) AS tasa_cancelacion
FROM tb_reservas
GROUP BY total_of_special_requests
ORDER BY total_of_special_requests ASC;

Conclusión: A medida que el cliente pide más cosas, cancela menos: la tasa baja de 47.72% sin solicitudes a 22.02% con 1, 17.86% con 3 y 5.00% con 5.
Quien no pide nada cancela más del doble que quien pide al menos una, lo que indica que las solicitudes reflejan compromiso con la estadía.
Con 4 y 5 solicitudes hay pocas reservas (340 y 40), así que esos porcentajes son menos confiables.

Hipótesis 3: ¿Cambiar la reserva es señal de compromiso? Al igual que en los ejercicios anteriores, utiliza CASE WHEN para segmentar la columna booking_changes en tres grupos: "Sin Cambios" (0), "1 a 3 cambios" y "Más de 3 cambios". Calcula la tasa de cancelación para cada grupo. ¿Quiénes tienen menor riesgo de cancelar?

In [0]:
%sql
SELECT CASE WHEN booking_changes = 0 THEN 'Sin Cambios'
            WHEN booking_changes BETWEEN 1 AND 3 THEN '1 a 3 cambios'
            ELSE 'Más de 3 cambios' END AS segmento_cambios,
       COUNT(*) AS total_reservas,
       ROUND(AVG(is_canceled), 4) AS tasa_cancelacion
FROM tb_reservas
GROUP BY CASE WHEN booking_changes = 0 THEN 'Sin Cambios'
              WHEN booking_changes BETWEEN 1 AND 3 THEN '1 a 3 cambios'
              ELSE 'Más de 3 cambios' END
ORDER BY tasa_cancelacion ASC;

Conclusión: El grupo de "1 a 3 cambios" tiene el menor riesgo de cancelar (15.59%), frente a 18.04% de "Más de 3 cambios" y 40.85% de "Sin Cambios".
Modificar la reserva sí es señal de compromiso: quien nunca la toca cancela 2.6 veces más que quien la ajusta de 1 a 3 veces.
El grupo de "Más de 3 cambios" tiene pocas reservas (643), por lo que su tasa debe tomarse con cautela.

Hipótesis 4: El factor precio (adr). Agrupa las reservas por tipo de hotel (hotel). Para cada tipo, muestra la cantidad total de cancelaciones y el precio promedio (adr) solo de las reservas canceladas. En base a tu consulta, ¿en qué tipo de hotel el precio promedio de cancelación es más alto?

In [0]:
%sql
SELECT hotel,
       COUNT(*) AS cantidad_cancelaciones,
       ROUND(AVG(adr), 2) AS adr_promedio_cancelado
FROM tb_reservas
WHERE is_canceled = 1
GROUP BY hotel
ORDER BY adr_promedio_cancelado DESC;

Conclusión: El Resort Hotel tiene el precio promedio de cancelación más alto (USD 105.79 por noche), frente a USD 104.69 del City Hotel.
La diferencia es de solo USD 1.10, así que el precio no distingue a un hotel del otro.
El City Hotel concentra 3 veces más cancelaciones (33,102 contra 11,122), por lo que la mayor pérdida de ingresos está ahí.